In [25]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


@dataclass
class MultiTBReaderConfig:
    """Configuración para extraer múltiples experimentos de TensorBoard."""
    tb_logs_dir: Path
    exp_pattern: str = "xgb_seed42_*"
    size_guidance: Dict[str, int] = field(
        default_factory=lambda: {"scalars": 0, "tensors": 0}
    )
    target_tags: Optional[List[str]] = None


def discover_experiment_event_dirs(config: MultiTBReaderConfig) -> List[Path]:
    """Busca recursivamente directorios con tfevents dentro de xgb_seed42_*.

    :param config: Configuración con ruta raíz y patrón glob.
    :return: Lista ordenada de directorios que contienen archivos tfevents.
    """
    if not config.tb_logs_dir.exists():
        raise FileNotFoundError(f"No existe la ruta: {config.tb_logs_dir}")

    exp_dirs = [
        d for d in config.tb_logs_dir.glob(config.exp_pattern) if d.is_dir()
    ]

    event_dirs = {
        f.parent
        for exp_dir in exp_dirs
        for f in exp_dir.rglob("events.out.tfevents.*")
    }
    return sorted(event_dirs)


def extract_single_dir(
    event_dir: Path,
    tb_logs_dir: Path,
    config: MultiTBReaderConfig,
) -> pd.DataFrame:
    """Extrae escalares de un directorio y registra su ruta relativa a tb_logs.

    :param event_dir: Directorio con archivos tfevents.
    :param tb_logs_dir: Directorio raíz tb_logs.
    :param config: Configuración de lectura.
    :return: DataFrame con los eventos del directorio.
    """
    acc = EventAccumulator(str(event_dir), size_guidance=config.size_guidance)
    acc.Reload()

    available_tags = acc.Tags().get("scalars", [])
    if not available_tags:
        return pd.DataFrame()

    tags_to_read = (
        [t for t in available_tags if t in config.target_tags]
        if config.target_tags is not None
        else available_tags
    )

    rel_path = str(event_dir.relative_to(tb_logs_dir))
    frames: List[pd.DataFrame] = []

    for tag in tags_to_read:
        events = acc.Scalars(tag)
        if not events:
            continue

        steps, wall_times, values = zip(
            *((e.step, e.wall_time, e.value) for e in events)
        )

        frames.append(
            pd.DataFrame({
                "rel_path": rel_path,
                "tag": tag,
                "step": np.asarray(steps, dtype=np.int64),
                "wall_time": np.asarray(wall_times, dtype=np.float64),
                "value": np.asarray(values, dtype=np.float32),
            })
        )

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_all_xgb_experiments(config: MultiTBReaderConfig) -> pd.DataFrame:
    """Carga todos los experimentos xgb_seed42_* y separa experimento y trial.

    :param config: Instancia de MultiTBReaderConfig.
    :return: DataFrame con columnas [experiment, trial, tag, step, wall_time, value].
    """
    event_dirs = discover_experiment_event_dirs(config)
    if not event_dirs:
        return pd.DataFrame()

    frames = [
        extract_single_dir(d, config.tb_logs_dir, config) for d in event_dirs
    ]
    valid_frames = [df for df in frames if not df.empty]

    if not valid_frames:
        return pd.DataFrame()

    full_df = pd.concat(valid_frames, ignore_index=True)

    # Separación vectorizada de "xgb_seed42_.../f1_t3_top1_..." en 2 columnas
    split_paths = full_df["rel_path"].str.split("/", n=1, expand=True)
    full_df["experiment"] = split_paths[0].astype("category")

    if split_paths.shape[1] > 1:
        full_df["trial"] = split_paths[1].fillna("root").astype("category")
    else:
        full_df["trial"] = pd.Series("root", index=full_df.index, dtype="category")

    full_df["tag"] = full_df["tag"].astype("category")
    return full_df.drop(columns=["rel_path"])


def summarize_all_experiments(df: pd.DataFrame) -> pd.DataFrame:
    """Resume las métricas agrupadas por experimento, trial y tag.

    :param df: DataFrame generado por load_all_xgb_experiments.
    :return: DataFrame con valores last, max, min y conteo de steps.
    """
    if df.empty:
        return pd.DataFrame()

    return (
        df.sort_values("step")
        .groupby(["experiment", "trial", "tag"], observed=True)["value"]
        .agg(last="last", max="max", min="min", steps="count")
        .reset_index()
    )

In [28]:
if __name__ == "__main__":
    tb_root = Path(
        "/home/andreoseo/Documents/MiSongGyny2026/src/"
        "gloss_bge_m3_ligthning/checkpoints/tb_logs"
    )

    cfg = MultiTBReaderConfig(
        tb_logs_dir=tb_root,
        exp_pattern="xgb_seed42_*",
    )

    df_all = load_all_xgb_experiments(cfg)
    df_summary = summarize_all_experiments(df_all)

    # Ver la tabla comparativa completa (Experimento vs Trial vs Métrica)
    
    # Opcional: Pivotar para tener cada métrica como columna
    df_pivot = df_summary.pivot(
        index=["experiment", "trial"],
        columns="tag",
        values="last",
    ).reset_index()
    
    r = df_summary[df_summary["tag"].str.contains("val_f1", na=False) 
        & df_summary["trial"].str.contains("top1", na=False)][["experiment","trial","tag","max"]]

    # Tomando en cuenta el orden de los experimentos fue
    # t1 no gloss
    # t1 yes gloss
    # t3 yes gloss
    # t3 no gloss
    print(r)

                    experiment                trial           tag       max
3   xgb_seed42_20261019_165749  f1_t1_top1_trial021  f1_t1/val_f1  0.812298
28  xgb_seed42_20261019_171151  f1_t1_top1_trial002  f1_t1/val_f1  0.832797
53  xgb_seed42_20261019_172508  f1_t3_top1_trial099  f1_t3/val_f1  0.832520
78  xgb_seed42_20261019_174314  f1_t3_top1_trial151  f1_t3/val_f1  0.791599


In [39]:
if __name__ == "__main__":
    tb_root = Path(
        "/home/andreoseo/Documents/MiSongGyny2026/src/"
        "gloss_bge_m3_ligthning/lightning_logs/"
    )

    cfg = MultiTBReaderConfig(
        tb_logs_dir=tb_root,
        exp_pattern="version_*",
    )

    df_all = load_all_xgb_experiments(cfg)
    df_summary = summarize_all_experiments(df_all)

    # Ver la tabla comparativa completa (Experimento vs Trial vs Métrica)
    
    # Opcional: Pivotar para tener cada métrica como columna
    df_pivot = df_summary.pivot(
        index=["experiment", "max"],
        columns="tag",
        values="last",
    ).reset_index()
    
    r = df_summary[df_summary["tag"].str.contains("val_f1", na=False) 
        & df_summary["experiment"].str.contains("_1[0-9]", na=False)][["experiment","trial","tag","max"]]

        # Condición para versiones 14 y 15 -> solo tag val_f1_t1
    cond_v14_15 = df_summary["experiment"].str.contains("_1[45]", na=False) & df_summary["tag"].str.contains("val_f1_t1", na=False)
    
    # Condición para versiones 16 y 17 -> solo tag val_f1_t3
    cond_v16_17 = df_summary["experiment"].str.contains("_1[67]", na=False) & df_summary["tag"].str.contains("val_f1_t3", na=False)
    
    # Unir ambas condiciones con OR (|) y seleccionar las columnas
    r = df_summary[cond_v14_15 | cond_v16_17][["experiment", "trial", "tag", "max"]]
    
    print(r)
    # Tomando en cuenta el orden de los experimentos fue
    # t1 no gloss - version 14
    # t1 yes gloss - version 15
    # t3 yes gloss - version 16
    # t3 no gloss - version 17
    #solo quiero mantener los tags acorde lo que si se entreno es decir para la version 14 y 15 el tag val_f1_t1
  

     experiment trial        tag       max
79   version_14  root  val_f1_t1  0.810858
96   version_15  root  val_f1_t1  0.847290
115  version_16  root  val_f1_t3  0.799309
132  version_17  root  val_f1_t3  0.790445


In [32]:
df_summary["tag"] ## val_f1_t1 y val_f1_t3
df_summary["experiment"] ## version 14 a la 17

,experiment,trial,tag,last,max,min,steps
0,version_10,root,epoch,0.000000e+00,0.000000,0.000000e+00,22
1,version_10,root,lr-AdamW/pg1,5.509642e-08,0.000005,5.509642e-08,20
2,version_10,root,lr-AdamW/pg2,1.101928e-05,0.000975,1.101928e-05,20
3,version_10,root,lr-AdamW/pg3,1.101928e-05,0.000975,1.101928e-05,20
4,version_10,root,lr-AdamW/pg4,1.101928e-05,0.000975,1.101928e-05,20
...,...,...,...,...,...,...,...
182,version_8,root,val_f1_t2,2.705648e-01,0.283000,2.506227e-01,7
183,version_8,root,val_f1_t3,2.330121e-01,0.286801,2.330121e-01,7
184,version_8,root,val_loss_t1,1.132871e+00,1.132871,4.976329e-01,7
185,version_8,root,val_loss_t2,9.400396e-01,0.940040,9.391646e-01,7
